# Gradients Training Demo

In this notebook, we take a general Qwen model and teach it how to answer biomedical research questions using PubMedQA medical data.

The journey is simple: give Gradients a dataset, let the network train the model, then test it on medical questions it has never seen before. At the end, you will see the base model and trained model side by side, with the expected answer from the test set.

No training scripts, no infrastructure work, no ML ops setup. Just an API key, a dataset, and a few notebook cells.

## Step 1: Install The Gradients SDK

One install gives this notebook everything it needs to launch training, sample data, load models, merge adapters, and run inference.

In [ ]:
%pip install -q --upgrade gradientsio==0.1.2

## Step 2: Choose The Mission

We will start with `Qwen/Qwen2.5-3B`, train it for 2 hours on normalized PubMedQA examples, and keep a separate PubMedQA test set untouched for the final showdown.

Paste your Gradients API key when asked. Everything else is already filled in.

In [ ]:
import os
from getpass import getpass
import gradientsio as gradients

MODEL_ID = "Qwen/Qwen2.5-3B"
TRAIN_DATASET = "gradients-io-tournaments/PubMedQA-Normalized-Train"
TEST_DATASET = "gradients-io-tournaments/PubMedQA-Normalized-Test"
RESULT_MODEL_NAME = "pubmedqa-qwen2-5-3b-gradients-demo"

os.environ["GRADIENTS_API_KEY"] = os.getenv("GRADIENTS_API_KEY") or getpass("Gradients API key: ").strip()
client = gradients.GradientsClient()

## Step 3: Send The Model To Training

This is the zero-faff part: we point Gradients at the medical training dataset, pick the base model, and launch the job.

The printed task ID is your receipt. If you close the notebook, paste that ID into the next step and continue where you left off.

In [ ]:
task = client.train(
    model=MODEL_ID,
    task_type=gradients.TaskType.INSTRUCT,
    hours=2,
    dataset=TRAIN_DATASET,
    field_instruction="instruction",
    field_input="input",
    field_output="output",
    result_model_name=RESULT_MODEL_NAME,
)

TASK_ID = task.task_id
print(f"Training task created: {TASK_ID}")

## Step 4: Wait For The Trained Model

Gradients now does the heavy lifting: dataset prep, scheduling, training, evaluation, and publishing the trained model repo.

Run this cell after launch. If you already have a task ID from an earlier run, paste it in and the notebook will wait for the final trained model.

In [ ]:
TASK_ID = globals().get("TASK_ID")
TRAINED_MODEL_REPO = client.tasks.handle(TASK_ID).wait().trained_model_repository

## Step 5: Pick Questions The Model Has Never Seen

Now we pull a few examples from the held-out test set. These are not part of the training run.

This is where the story gets interesting: can a small base model learn the medical answer style from your custom data, then apply it to fresh biomedical questions?

In [ ]:
samples = gradients.load_dataset_rows(TEST_DATASET)

def build_prompt(row):
    return f"{(row.get('instruction') or '').strip()}\n\nAnswer:"

for index, row in enumerate(samples, start=1):
    print(f"Example {index}: {row.get('instruction')}")

## Step 6: Watch The Before And After

Time for the proof.

We ask the original base model and the newly trained model the same unseen medical questions. Then we place both answers next to the expected PubMedQA answer so the improvement is easy to judge at a glance.

This is the whole Gradients loop: bring your data, train your model, test the difference.

In [ ]:
from IPython.display import Markdown, display

samples = gradients.load_dataset_rows(TEST_DATASET)
prompts = [build_prompt(row) for row in samples]

sampler = gradients.ModelSampler()
base = sampler.generate(MODEL_ID, prompts)
trained = sampler.generate_with_adapter(TRAINED_MODEL_REPO, prompts, base_model_repo=MODEL_ID)

for index, (row, prompt, base_answer, trained_answer) in enumerate(zip(samples, prompts, base, trained), start=1):
    display(Markdown(f"""
---
## Example {index} | PubMed ID: {row.get('pubid')}
### Question
{prompt.removesuffix('Answer:').strip()}
### Expected Answer
{(row.get('output') or '').strip()}
### Trained Model Answer
{trained_answer.strip()}
### Base Model Answer
{base_answer.strip()}
"""))